In [12]:
from pathlib import Path
import pandas as pd
import glob
import os 
import cv2
from tqdm import tqdm
import re 

In [14]:
base = Path(r"R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track")
mouse_num = "994"

# Equivalent to: */994*/My_WebCam/*.avi
avi_matches = sorted(base.glob(f"*/{mouse_num}*/My_WebCam/*.avi"))

# Get unique My_WebCam folders that actually contain AVI files
webcam_dirs = sorted({p.parent for p in avi_matches})

print(f"Found {len(webcam_dirs)} My_WebCam folders for mouse {mouse_num}:")
for d in webcam_dirs:
    print(d)
def natural_key(path):
    """
    Sorts 0.avi, 1.avi, 2.avi, ..., 10.avi correctly.
    Also works for behavCam00.avi, behavCam01.avi, etc.
    """
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]
rows = []
for d in webcam_dirs:
    avi_files = sorted(d.glob("*.avi"), key=natural_key)
    mp4_files = sorted(d.glob("*.mp4"), key=natural_key)
    csv_files = sorted(d.glob("*.csv"), key=natural_key)
    concat_files = sorted(d.glob("*concactenated*.mp4"), key=natural_key)
    rows.append({
        "date_folder": d.parent.parent.name,
        "session_folder": d.parent.name,
        "My_WebCam_path": str(d),
        "n_avi": len(avi_files),
        "n_mp4": len(mp4_files),
        "n_csv": len(csv_files),
        "n_concactenated": len(concat_files),
        "first_avi": avi_files[0].name if avi_files else None,
        "last_avi": avi_files[-1].name if avi_files else None,
        "concactenated_file": concat_files[0].name if concat_files else None,
    })
webcam_summary = pd.DataFrame(rows)
webcam_summary.head()

Found 15 My_WebCam folders for mouse 994:
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_31\994_16_40_42\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_01\994_17_21_52\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_02\994_18_10_53\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_03\994_17_30_10\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_04\994_17_06_43\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_04\994_17_19_27\My_WebCam
R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_da

,date_folder,session_folder,My_WebCam_path,n_avi,n_mp4,n_csv,n_concactenated,first_avi,last_avi,concactenated_file
0,2024_12_30,994_18_25_08,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,18,1,1,1,behavCam00.avi,behavCam17.avi,m994_30122024_18_25_08_concactenatedbehavCam00...
1,2024_12_31,994_16_40_42,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,16,0,1,0,0.avi,15.avi,NaN
2,2025_01_01,994_17_21_52,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,18,0,1,0,0.avi,17.avi,NaN
3,2025_01_02,994_18_10_53,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,18,0,1,0,0.avi,17.avi,NaN
4,2025_01_03,994_17_30_10,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,18,0,1,0,0.avi,17.avi,NaN


In [15]:
mouse_num = "994"

def natural_key(path):
    """
    Sorts:
        0.avi, 1.avi, 2.avi, ..., 10.avi
    and:
        behavCam00.avi, behavCam01.avi, ..., behavCam10.avi
    in numerical order.
    """
    path = Path(path)
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]


def normalize_avi_names(session_path):
    """
    Rename:
        0.avi              -> behavCam00.avi
        1.avi              -> behavCam01.avi
        behavCam0.avi      -> behavCam00.avi
        behavCam00.avi     -> unchanged
    """
    session_path = Path(session_path)

    avi_files = sorted(session_path.glob("*.avi"), key=natural_key)

    for video_f in avi_files:
        stem = video_f.stem  # safer than strip(".avi")

        if stem.startswith("behavCam"):
            num_part = stem.replace("behavCam", "")
        else:
            num_part = stem

        if not num_part.isdigit():
            print(f"Skipping rename for unexpected AVI name: {video_f.name}")
            continue

        reformatted_name = f"behavCam{int(num_part):02d}.avi"
        reformatted_path = video_f.with_name(reformatted_name)

        if video_f.name == reformatted_name:
            continue

        if reformatted_path.exists():
            print(f"Target already exists, not renaming: {video_f.name} -> {reformatted_name}")
            continue

        print(f"Renaming: {video_f.name} -> {reformatted_name}")
        video_f.rename(reformatted_path)


def make_session_name(date_folder, session_folder, mouse_num):
    """
    Example:
        date_folder    = 2024_12_30
        session_folder = 994_18_25_08

    Output:
        m994_30122024_18_25_08
    """
    yyyy, mm, dd = date_folder.split("_")
    ddmmyyyy = f"{dd}{mm}{yyyy}"

    session_parts = session_folder.split("_")
    time_part = "_".join(session_parts[1:])

    return f"m{mouse_num}_{ddmmyyyy}_{time_part}"


def concatenate_session_avis(session_path, session_to_concat, overwrite=False):
    session_path = Path(session_path)

    normalize_avi_names(session_path)

    videos = sorted(session_path.glob("*.avi"), key=natural_key)

    if len(videos) == 0:
        print(f"No AVI files found in: {session_path}")
        return None

    first_stem = videos[0].stem
    last_stem = videos[-1].stem

    output_path = session_path / f"{session_to_concat}_concactenated{first_stem}_{last_stem}.mp4"

    if output_path.exists() and not overwrite:
        print(f"Output already exists, skipping: {output_path.name}")
        return output_path

    vcap = cv2.VideoCapture(str(videos[0]))

    if not vcap.isOpened():
        print(f"Could not open first video: {videos[0]}")
        return None

    width = int(vcap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(vcap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = vcap.get(cv2.CAP_PROP_FPS)
    vcap.release()

    if fps is None or fps <= 0:
        print(f"Bad FPS detected for {videos[0]}, using fps=30")
        fps = 30

    print("")
    print(f"Concatenating session: {session_to_concat}")
    print(f"Folder: {session_path}")
    print(f"Number of AVI files: {len(videos)}")
    print(f"Output: {output_path.name}")
    print(f"Width x Height: {width} x {height}, FPS: {fps}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (width, height)
    )

    if not video_writer.isOpened():
        print(f"Could not create output video: {output_path}")
        return None

    for v in tqdm(videos, desc=session_to_concat):
        curr_v = cv2.VideoCapture(str(v))

        if not curr_v.isOpened():
            print(f"Could not open video, skipping: {v}")
            continue

        while True:
            r, frame = curr_v.read()

            if not r:
                break

            video_writer.write(frame)

        curr_v.release()

    video_writer.release()

    print(f"Finished: {output_path}")
    return output_path

In [16]:
outputs = []

for idx, row in webcam_summary.iterrows():
    session_path = Path(row["My_WebCam_path"])
    date_folder = row["date_folder"]
    session_folder = row["session_folder"]

    session_to_concat = make_session_name(
        date_folder=date_folder,
        session_folder=session_folder,
        mouse_num=mouse_num
    )

    output_path = concatenate_session_avis(
        session_path=session_path,
        session_to_concat=session_to_concat,
        overwrite=False
    )

    outputs.append({
        "date_folder": date_folder,
        "session_folder": session_folder,
        "My_WebCam_path": str(session_path),
        "output_path": str(output_path) if output_path is not None else None,
    })

concat_results = pd.DataFrame(outputs)

concat_results

Output already exists, skipping: m994_30122024_18_25_08_concactenatedbehavCam00_behavCam17.mp4
Renaming: 0.avi -> behavCam00.avi
Renaming: 1.avi -> behavCam01.avi
Renaming: 2.avi -> behavCam02.avi
Renaming: 3.avi -> behavCam03.avi
Renaming: 4.avi -> behavCam04.avi
Renaming: 5.avi -> behavCam05.avi
Renaming: 6.avi -> behavCam06.avi
Renaming: 7.avi -> behavCam07.avi
Renaming: 8.avi -> behavCam08.avi
Renaming: 9.avi -> behavCam09.avi
Renaming: 10.avi -> behavCam10.avi
Renaming: 11.avi -> behavCam11.avi
Renaming: 12.avi -> behavCam12.avi
Renaming: 13.avi -> behavCam13.avi
Renaming: 14.avi -> behavCam14.avi
Renaming: 15.avi -> behavCam15.avi

Concatenating session: m994_31122024_16_40_42
Folder: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_31\994_16_40_42\My_WebCam
Number of AVI files: 16
Output: m994_31122024_16_40_42_concactenatedbehavCam00_behavCam15.mp4
Width x Height: 640 x 480, FPS: 60.0


m994_31122024_16_40_42: 100%|█████████████████████████████████████| 16/16 [00:42<00:00,  2.64s/it]


Finished: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_31\994_16_40_42\My_WebCam\m994_31122024_16_40_42_concactenatedbehavCam00_behavCam15.mp4
Renaming: 0.avi -> behavCam00.avi
Renaming: 1.avi -> behavCam01.avi
Renaming: 2.avi -> behavCam02.avi
Renaming: 3.avi -> behavCam03.avi
Renaming: 4.avi -> behavCam04.avi
Renaming: 5.avi -> behavCam05.avi
Renaming: 6.avi -> behavCam06.avi
Renaming: 7.avi -> behavCam07.avi
Renaming: 8.avi -> behavCam08.avi
Renaming: 9.avi -> behavCam09.avi
Renaming: 10.avi -> behavCam10.avi
Renaming: 11.avi -> behavCam11.avi
Renaming: 12.avi -> behavCam12.avi
Renaming: 13.avi -> behavCam13.avi
Renaming: 14.avi -> behavCam14.avi
Renaming: 15.avi -> behavCam15.avi
Renaming: 16.avi -> behavCam16.avi
Renaming: 17.avi -> behavCam17.avi

Concatenating session: m994_01012025_17_21_52
Folder: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_01\994_17_21_52\My_WebCam
N

m994_01012025_17_21_52: 100%|█████████████████████████████████████| 18/18 [00:46<00:00,  2.58s/it]


Finished: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_01\994_17_21_52\My_WebCam\m994_01012025_17_21_52_concactenatedbehavCam00_behavCam17.mp4
Renaming: 0.avi -> behavCam00.avi
Renaming: 1.avi -> behavCam01.avi
Renaming: 2.avi -> behavCam02.avi
Renaming: 3.avi -> behavCam03.avi
Renaming: 4.avi -> behavCam04.avi
Renaming: 5.avi -> behavCam05.avi
Renaming: 6.avi -> behavCam06.avi
Renaming: 7.avi -> behavCam07.avi
Renaming: 8.avi -> behavCam08.avi
Renaming: 9.avi -> behavCam09.avi
Renaming: 10.avi -> behavCam10.avi
Renaming: 11.avi -> behavCam11.avi
Renaming: 12.avi -> behavCam12.avi
Renaming: 13.avi -> behavCam13.avi
Renaming: 14.avi -> behavCam14.avi
Renaming: 15.avi -> behavCam15.avi
Renaming: 16.avi -> behavCam16.avi
Renaming: 17.avi -> behavCam17.avi

Concatenating session: m994_02012025_18_10_53
Folder: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_02\994_18_10_53\My_WebCam
N

...

m994_09012025_20_39_39: 100%|█████████████████████████████████████| 14/14 [00:37<00:00,  2.71s/it]


Finished: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_09\994_20_39_39\My_WebCam\m994_09012025_20_39_39_concactenatedbehavCam00_behavCam13.mp4
Renaming: 0.avi -> behavCam00.avi
Renaming: 1.avi -> behavCam01.avi
Renaming: 2.avi -> behavCam02.avi
Renaming: 3.avi -> behavCam03.avi
Renaming: 4.avi -> behavCam04.avi
Renaming: 5.avi -> behavCam05.avi
Renaming: 6.avi -> behavCam06.avi
Renaming: 7.avi -> behavCam07.avi
Renaming: 8.avi -> behavCam08.avi
Renaming: 9.avi -> behavCam09.avi
Renaming: 10.avi -> behavCam10.avi
Renaming: 11.avi -> behavCam11.avi
Renaming: 12.avi -> behavCam12.avi
Renaming: 13.avi -> behavCam13.avi
Renaming: 14.avi -> behavCam14.avi
Renaming: 15.avi -> behavCam15.avi
Renaming: 16.avi -> behavCam16.avi
Renaming: 17.avi -> behavCam17.avi

Concatenating session: m994_10012025_18_33_55
Folder: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_10\994_18_33_55\My_WebCam
N

m994_10012025_18_33_55: 100%|█████████████████████████████████████| 18/18 [00:46<00:00,  2.59s/it]

Finished: R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2025_01_10\994_18_33_55\My_WebCam\m994_10012025_18_33_55_concactenatedbehavCam00_behavCam17.mp4


,date_folder,session_folder,My_WebCam_path,output_path
0,2024_12_30,994_18_25_08,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
1,2024_12_31,994_16_40_42,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
2,2025_01_01,994_17_21_52,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
3,2025_01_02,994_18_10_53,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
4,2025_01_03,994_17_30_10,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
5,2025_01_04,994_17_06_43,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
6,2025_01_04,994_17_19_27,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
7,2025_01_04,994_17_39_48,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
8,2025_01_05,994_17_40_18,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...
9,2025_01_05,994_17_42_59,R:\Basic_Sciences\Phys\ContractorLab\Projects\...,R:\Basic_Sciences\Phys\ContractorLab\Projects\...


In [4]:
## old code below here 


session_to_concat = 'm994_30122024_18_25_08'

session_path=r'R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam'

fList = glob.glob(session_path+'/*.avi')

fList

## justify digits at end of fname with 00s
for video_f in fList:
    fname = video_f.split(os.sep)[-1].strip('.avi')
    reformatted = 'behavCam'+fname.strip('behavCam').zfill(2) +'.avi'
    os.rename(video_f, '/'.join(video_f.split(os.sep)[:-1])+'/'+reformatted)

videos = sorted(glob.glob(session_path+'/*.avi'))

print(videos)


vstoconcat = videos

vcap = cv2.VideoCapture(vstoconcat[0])
width  = vcap.get(cv2.CAP_PROP_FRAME_WIDTH)
height  = vcap.get(cv2.CAP_PROP_FRAME_HEIGHT)
fps = vcap.get(cv2.CAP_PROP_FPS)
# Create a new video
video = cv2.VideoWriter('/'.join(vstoconcat[0].split(os.sep)[:-1])+'/'+ session_to_concat + "_concactenated" + vstoconcat[0].split(os.sep)[-1].strip('.avi') + "_" + vstoconcat[-1].split(os.sep)[-1].strip('.avi') + ".mp4", 
                        cv2.VideoWriter_fourcc(*"MPEG"), fps, (int(width), int(height)))
# Write all the frames sequentially to the new video
for v in tqdm(vstoconcat):
    print(v)
    curr_v = cv2.VideoCapture(v)
    while curr_v.isOpened():
        r, frame = curr_v.read()    # Get return value and curr frame of curr video
        if not r:
            break
        video.write(frame)          # Write the frame
video.release() 

['R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Miniscope_data\\Miniscope_data\\Linear_track\\2024_12_30\\994_18_25_08\\My_WebCam\\behavCam00.avi', 'R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Miniscope_data\\Miniscope_data\\Linear_track\\2024_12_30\\994_18_25_08\\My_WebCam\\behavCam01.avi', 'R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Miniscope_data\\Miniscope_data\\Linear_track\\2024_12_30\\994_18_25_08\\My_WebCam\\behavCam02.avi', 'R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Miniscope_data\\Miniscope_data\\Linear_track\\2024_12_30\\994_18_25_08\\My_WebCam\\behavCam03.avi', 'R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Miniscope_data\\Miniscope_data\\Linear_track\\2024_12_30\\994_18_25_08\\My_WebCam\\behavCam04.avi', 'R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Miniscope_data\\Miniscope_data\\Linear_track\\2024_12_30\\994_18_25_08\\My_WebCam\\behavCam05.avi', 'R:\\Basic_Sciences\\Phys\\ContractorLab\\Projects\\YZ\\Minisco

  0%|                                                                      | 0/18 [00:00<?, ?it/s]

R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam\behavCam00.avi


  6%|███▍                                                          | 1/18 [00:03<00:52,  3.07s/it]

R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam\behavCam01.avi


...

 89%|██████████████████████████████████████████████████████▏      | 16/18 [00:50<00:06,  3.14s/it]

R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam\behavCam16.avi


 94%|█████████████████████████████████████████████████████████▌   | 17/18 [00:53<00:03,  3.15s/it]

R:\Basic_Sciences\Phys\ContractorLab\Projects\YZ\Miniscope_data\Miniscope_data\Linear_track\2024_12_30\994_18_25_08\My_WebCam\behavCam17.avi


100%|█████████████████████████████████████████████████████████████| 18/18 [00:53<00:00,  3.00s/it]


In [ ]:
## renaming timestampfiles from shell script
#!bash copyAndRenameTimeStampFiles.sh

In [ ]:
# to move all final concactenated file names
#!base="/Volumes/fsmresfiles/Basic_Sciences/Phys/ContractorLab/Projects/YZ/Miniscope_data/Miniscope_data/Linear_track"; mouse_num="328"; dest="$base/BehavCamConcactenated_${mouse_num}"; mkdir -p "$dest"; find "$base" -type f -path "*/${mouse_num}_*/My_WebCam/*concactenatedbehav*.mp4" -exec cp -n {} "$dest"/ \;